# Tahap 3: Case Retrieval
## Alur Kerja (sesuai ketentuan tugas)
1. **Representasi Vektor** — dua pendekatan dibangun dan dibandingkan:
   - **Jalur A (Statistik):** TF-IDF + model ML (SVM, Naive Bayes) untuk classification/retrieval
   - **Jalur B (Embedding):** IndoBERT pretrained + cosine similarity untuk retrieval
2. **Splitting Data** — train:test = 80:20
3. **Fungsi Retrieval** — `retrieve(query, k=5)` dibangun untuk masing-masing jalur
4. **Pengujian Awal** — 5-10 query uji dengan ground-truth case_id, disimpan di `data/eval/queries.json`



## 0. Setup & Path

In [ ]:
# Jalankan sekali saja jika library belum terinstall
# !pip install scikit-learn pandas numpy
# Untuk Jalur B (embedding), tambahan:
# !pip install transformers torch sentencepiece

In [ ]:
!pip install scikit-learn pandas numpy transformers torch sentencepiece 

In [ ]:
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

BASE_DIR = Path(".").resolve().parent  

PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed dir : {PROCESSED_DIR}")
print(f"Eval dir      : {EVAL_DIR}")


Processed dir : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\processed
Eval dir      : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\eval


In [ ]:
import re

def remove_admin_identity_lines(text: str) -> str:
    """
    Hapus baris yang HANYA berisi label field identitas administratif Terdakwa (Nama Lengkap,
    Tempat Lahir, Umur/Tanggal Lahir, dst), dengan/tanpa nilai pendek di baris yang sama.

    LATAR BELAKANG: pada dokumen putusan MA RI, narasi dakwaan (fakta perbuatan) sering bersebelahan
    langsung dengan blok identitas administratif Terdakwa dalam hasil ekstraksi PDF/OCR, sehingga
    ringkasan_fakta dari Tahap 2 (yang diambil sebagai potongan teks mulai dari kalimat pembuka
    dakwaan) bisa ikut menangkap blok identitas tersebut. Fungsi ini menghapus baris-baris semacam
    itu SEBELUM teks dipakai sebagai representasi vektor, supaya representasi lebih fokus ke fakta
    perbuatan yang sebenarnya membedakan satu kasus dari kasus lain - bukan identitas administratif
    yang hampir selalu berstruktur sama di semua dokumen (sehingga tidak diskriminatif untuk retrieval).

    Pendekatan ini sengaja TIDAK 100% sempurna (beberapa residu nilai field tanpa label mungkin masih
    tersisa) - regex yang lebih agresif berisiko menghapus konten substantif yang kebetulan mirip
    pola. Trade-off ini diterima karena dampak residu kecil terhadap representasi vektor TF-IDF minim.
    """
    field_label_pattern = re.compile(
        r"^\s*(?:I+\.?\s*|U\.?\s*)?"  
        r"(?:Nama\s+[Ll]engkap|Tempat\s+[Ll]ahir|Umur\s*/\s*[Tt]anggal\s+[Ll]ahir|Jenis\s+[Kk]elamin|"
        r"Kebangsaan|Tempat\s+[Tt]inggal|Agama|Pekerjaan|Pendidikan)"
        r"\s*:?\s*[\w\s\-/\.,]{0,3}\s*$",
        flags=re.IGNORECASE | re.MULTILINE,
    )
    cleaned = field_label_pattern.sub("", text)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)  
    return cleaned


In [ ]:
# Muat data hasil Tahap 2
cases_path = PROCESSED_DIR / "cases.csv"
cases_df = pd.read_csv(cases_path)

print(f"Jumlah kasus dimuat: {len(cases_df)}")
assert len(cases_df) >= 30, "Jumlah dokumen kurang dari 30 - cek kembali hasil Tahap 2!"

ringkasan_fakta_clean = cases_df["ringkasan_fakta"].fillna("").apply(remove_admin_identity_lines)
cases_df["text_for_vector"] = ringkasan_fakta_clean + " " + cases_df["argumen_hukum"].fillna("")

# Validasi: pastikan tidak ada teks kosong yang akan merusak representasi vektor
n_empty = (cases_df["text_for_vector"].str.strip() == "").sum()
if n_empty > 0:
    print(f"⚠️  {n_empty} dokumen punya text_for_vector kosong - cek ulang hasil Tahap 2 untuk dokumen ini!")
else:
    print("✅ Semua dokumen punya representasi teks yang siap divektorkan.")

cases_df[["case_id", "pasal_terbukti"]].head()


Jumlah kasus dimuat: 30
✅ Semua dokumen punya representasi teks yang siap divektorkan.


,case_id,pasal_terbukti_draft
0,case_001,Pasal 30 ayat (1) jo Pasal 46
1,case_002,Pasal 51 ayat (1)
2,case_003,Pasal 45
3,case_004,Pasal 27 ayat (3) jo. Pasal 45 ayat (3)
4,case_005,Pasal 27 ayat (3) jo. Pasal 45 ayat (3)


## 1. Splitting Data (80:20)

Data dibagi menjadi train (80%) dan test (20%). Karena `pasal_terbukti_draft` dipakai sebagai target
klasifikasi pada Jalur A, split dilakukan dengan **stratifikasi** jika memungkinkan (supaya proporsi
tiap kelas pasal terjaga); jika ada kelas dengan hanya 1 anggota (umum terjadi pada dataset kecil),
otomatis fallback ke split acak biasa.


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("=== Distribusi pasal_terbukti (target klasifikasi) ===")
print(cases_df["pasal_terbukti"].value_counts())

class_counts = cases_df["pasal_terbukti"].value_counts()
can_stratify = (class_counts >= 2).all()
print(f"\nBisa stratifikasi? {can_stratify} (stratifikasi butuh setiap kelas punya >= 2 anggota)")


=== Distribusi pasal_terbukti_draft (target klasifikasi) ===
pasal_terbukti_draft
Pasal 2 ayat (1)                                      4
Pasal 27                                              3
Pasal 45                                              3
Pasal 27 ayat (3) jo. Pasal 45 ayat (3)               2
Pasal 28 ayat (1) jo. Pasal 45                        2
Pasal 45 ayat (1) jo Pasal 27 ayat (1)                1
Pasal 51 ayat (1)                                     1
Pasal 30 ayat (1) jo Pasal 46                         1
Pasal 84 ayat (2)                                     1
Pasal 2                                               1
Pasal 480                                             1
Pasal 45 ayat (4) Jo Pasal 27 ayat (4)                1
Pasal 28 ayat (1) Jo. Pasal 45                        1
Pasal 36 Jo Pasal 30 ayat (3)                         1
Pasal 27 ayat (4) Jo. Pasal 45 ayat (4)               1
Pasal 27 ayat (1) jo Pasal 45 ayat (1)                1
Pasal 32 ayat (2) jo P

In [22]:
stratify_col = cases_df["pasal_terbukti"] if can_stratify else None

train_df, test_df = train_test_split(
    cases_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_col,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Jumlah data train : {len(train_df)}")
print(f"Jumlah data test  : {len(test_df)}")
print(f"\nCase ID di test set: {list(test_df['case_id'])}")


Jumlah data train : 24
Jumlah data test  : 6

Case ID di test set: ['case_028', 'case_016', 'case_024', 'case_018', 'case_009', 'case_010']


## 2. Jalur A — Representasi TF-IDF

Membangun representasi vektor TF-IDF dari seluruh corpus (train set, agar tidak terjadi data leakage
ke test set), lalu melatih model klasifikasi (SVM dan Naive Bayes) sebagai pembanding performa.


In [23]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),  # unigram + bigram, menangkap frasa hukum seperti "pidana penjara"
    min_df=1,
)

# Fit HANYA pada train set untuk menghindari data leakage
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["text_for_vector"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["text_for_vector"])

print(f"Ukuran matriks TF-IDF train : {X_train_tfidf.shape}")
print(f"Ukuran matriks TF-IDF test  : {X_test_tfidf.shape}")
print(f"Jumlah fitur (kata/frasa)   : {len(tfidf_vectorizer.get_feature_names_out())}")


Ukuran matriks TF-IDF train : (24, 2000)
Ukuran matriks TF-IDF test  : (6, 2000)
Jumlah fitur (kata/frasa)   : 2000


In [ ]:
y_train = train_df["pasal_terbukti"]
y_test = test_df["pasal_terbukti"]

# Model 1: SVM
svm_model = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE)
svm_model.fit(X_train_tfidf, y_train)
svm_train_acc = svm_model.score(X_train_tfidf, y_train)
print(f"SVM - akurasi pada train set: {svm_train_acc:.2%}")

# Model 2: Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_train_acc = nb_model.score(X_train_tfidf, y_train)
print(f"Naive Bayes - akurasi pada train set: {nb_train_acc:.2%}")



SVM - akurasi pada train set: 62.50%
Naive Bayes - akurasi pada train set: 41.67%


C:\Users\maull\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\utils\multiclass.py:213: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  y_type = type_of_target(y, input_name="y")
C:\Users\maull\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\maull\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\preprocessing\_label.py:302: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  self.y_type_ = type_of_target(y, input_name="y")
C:\Users\maull\AppData\Local\Packages\PythonSoftwareFoun

## 3. Fungsi Retrieval — Jalur A (TF-IDF + Cosine Similarity)

Sesuai spesifikasi tugas, fungsi `retrieve()` menerima query teks bebas dan mengembalikan top-k
`case_id` yang paling mirip, berdasarkan cosine similarity terhadap representasi TF-IDF seluruh
case base (train set).


In [25]:
def retrieve_tfidf(query: str, k: int = 5) -> list[str]:
    """
    Fungsi retrieval berbasis TF-IDF + cosine similarity.

    Args:
        query: teks kasus baru (bisa berupa ringkasan fakta atau deskripsi kasus)
        k: jumlah top-k case_id paling mirip yang dikembalikan

    Returns:
        List case_id (dari train set / case base), diurutkan dari paling mirip
    """
    # 1) Pre-process query - TfidfVectorizer sudah menangani lowercasing & tokenisasi secara internal
    # 2) Hitung vektor query menggunakan vectorizer yang SAMA dengan yang sudah di-fit pada case base
    query_vector = tfidf_vectorizer.transform([query])

    # 3) Hitung cosine similarity dengan semua case vectors (train set = case base)
    similarities = cosine_similarity(query_vector, X_train_tfidf).flatten()

    # 4) Kembalikan top-k case_id berdasarkan similarity tertinggi
    top_k_idx = np.argsort(similarities)[::-1][:k]
    top_k_case_ids = train_df.iloc[top_k_idx]["case_id"].tolist()

    return top_k_case_ids


In [26]:
# Uji cepat fungsi retrieve_tfidf dengan salah satu dokumen test set sebagai query
sample_query = test_df.iloc[0]["text_for_vector"]
sample_case_id = test_df.iloc[0]["case_id"]

result = retrieve_tfidf(sample_query, k=5)
print(f"Query dari case_id: {sample_case_id}")
print(f"Top-5 hasil retrieval (TF-IDF): {result}")


Query dari case_id: case_028
Top-5 hasil retrieval (TF-IDF): ['case_029', 'case_030', 'case_019', 'case_007', 'case_001']


## 4. Jalur B — Representasi Text Embedding (IndoBERT)

Membangun representasi vektor menggunakan model Transformer pre-trained Bahasa Indonesia
(`indobenchmark/indobert-base-p1`). Embedding diambil dari mean-pooling token terakhir (pendekatan
umum untuk representasi kalimat/dokumen dari model BERT yang tidak dilatih khusus untuk sentence
embedding).

> **Catatan performa:** proses ini mengunduh model (~500MB) saat pertama dijalankan dan membutuhkan
> waktu komputasi lebih lama dibanding TF-IDF, terutama tanpa GPU. Untuk 30 dokumen, total waktu proses
> biasanya tetap dalam orde menit (bukan jam) pada CPU standar.


In [27]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "indobenchmark/indobert-base-p1"

print(f"Memuat tokenizer & model: {MODEL_NAME} (unduhan pertama kali bisa beberapa menit)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()  # mode evaluasi (non-training), nonaktifkan dropout dll

device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = bert_model.to(device)
print(f"Model dimuat. Device yang dipakai: {device}")


Memuat tokenizer & model: indobenchmark/indobert-base-p1 (unduhan pertama kali bisa beberapa menit)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 38504.71it/s]

Model dimuat. Device yang dipakai: cpu


In [28]:
def get_bert_embedding(text: str, max_length: int = 512) -> np.ndarray:
    """
    Hitung embedding kalimat/dokumen dari teks menggunakan mean-pooling atas hidden state
    token terakhir IndoBERT. Teks dipotong ke max_length token pertama jika terlalu panjang
    (keterbatasan umum model BERT - 512 token).
    """
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=max_length, padding=True
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    # Mean-pooling: rata-rata embedding semua token (dengan mask untuk token padding)
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_embeddings = token_embeddings * attention_mask
    summed = masked_embeddings.sum(dim=1)
    counts = attention_mask.sum(dim=1)
    mean_pooled = summed / counts

    return mean_pooled.cpu().numpy().flatten()


In [29]:
from tqdm import tqdm

print("Menghitung embedding untuk seluruh train set (case base)...")
train_embeddings = np.vstack([
    get_bert_embedding(text) for text in tqdm(train_df["text_for_vector"], desc="Embedding train set")
])

print(f"Ukuran matriks embedding train: {train_embeddings.shape}")


Menghitung embedding untuk seluruh train set (case base)...


Embedding train set: 100%|██████████| 24/24 [00:05<00:00,  4.32it/s]

Ukuran matriks embedding train: (24, 768)


## 5. Fungsi Retrieval — Jalur B (IndoBERT Embedding + Cosine Similarity)

Struktur fungsi identik dengan `retrieve_tfidf()` agar mudah dibandingkan, hanya berbeda pada cara
menghitung representasi vektor query dan case base.


In [30]:
def retrieve_embedding(query: str, k: int = 5) -> list[str]:
    """
    Fungsi retrieval berbasis IndoBERT embedding + cosine similarity.

    Args:
        query: teks kasus baru
        k: jumlah top-k case_id paling mirip yang dikembalikan

    Returns:
        List case_id (dari train set / case base), diurutkan dari paling mirip
    """
    # 1) Pre-process query - ditangani otomatis oleh tokenizer BERT (termasuk subword tokenization)
    # 2) Hitung vektor query menggunakan model embedding yang sama dengan case base
    query_vector = get_bert_embedding(query).reshape(1, -1)

    # 3) Hitung cosine similarity dengan semua case vectors (train set = case base)
    similarities = cosine_similarity(query_vector, train_embeddings).flatten()

    # 4) Kembalikan top-k case_id berdasarkan similarity tertinggi
    top_k_idx = np.argsort(similarities)[::-1][:k]
    top_k_case_ids = train_df.iloc[top_k_idx]["case_id"].tolist()

    return top_k_case_ids


In [31]:
# Uji cepat fungsi retrieve_embedding dengan query yang sama seperti sebelumnya, untuk dibandingkan
result_embedding = retrieve_embedding(sample_query, k=5)
print(f"Query dari case_id: {sample_case_id}")
print(f"Top-5 hasil retrieval (TF-IDF)    : {result}")
print(f"Top-5 hasil retrieval (Embedding) : {result_embedding}")


Query dari case_id: case_028
Top-5 hasil retrieval (TF-IDF)    : ['case_029', 'case_030', 'case_019', 'case_007', 'case_001']
Top-5 hasil retrieval (Embedding) : ['case_007', 'case_023', 'case_015', 'case_026', 'case_006']


## 6. Pengujian Awal: Siapkan Query Uji + Ground Truth

Sesuai ketentuan tugas, disiapkan 5-10 query uji beserta ground-truth `case_id`. Strategi: setiap
dokumen di **test set** dijadikan satu query (menggunakan `text_for_vector`-nya sendiri), dengan asumsi
ground-truth sederhana berupa **dirinya sendiri** (`case_id` dari test set yang bersangkutan) — pendekatan
umum untuk evaluasi self-retrieval pada CBR ketika tidak ada anotator independen untuk menilai relevansi.

> **Catatan keterbatasan:** karena ground-truth di sini adalah "dokumen itu sendiri", metrik yang
> dihasilkan nanti (Tahap 5) sebenarnya mengukur **kemampuan sistem menemukan dokumen yang paling mirip
> dengan query yang berasal dari distribusi serupa**, bukan relevansi hukum yang dinilai pakar. Ini
> keterbatasan wajar untuk skala tugas kuliah - idealnya ground-truth dibuat oleh pakar yang menilai
> kasus mana yang benar-benar relevan secara hukum, bukan hanya kemiripan teks.


In [ ]:
queries_data = []

for _, row in test_df.iterrows():
    # Bersihkan query_text: hapus blok identitas administratif
    # (sama seperti yang dilakukan pada text_for_vector di case base)
    query_text_clean = remove_admin_identity_lines(row["text_for_vector"])

    pasal_q = row["pasal_terbukti"]
    same_pasal_train_ids = train_df[
        train_df["pasal_terbukti"] == pasal_q
    ]["case_id"].tolist()

    queries_data.append({
        "query_id"                       : f"query_{row['case_id']}",
        "query_text"                     : query_text_clean,
        "ground_truth_pasal"             : pasal_q,
        "ground_truth_case_ids_same_pasal": same_pasal_train_ids,
        # Catatan: case_id berikut ada di TEST SET, BUKAN di case base.
        # Tidak dipakai untuk evaluasi retrieval, tapi disimpan untuk
        # keperluan tracing/debugging.
        "_test_set_case_id"              : row["case_id"],
    })

queries_path = EVAL_DIR / "queries.json"
with open(queries_path, "w", encoding="utf-8") as f:
    json.dump(queries_data, f, ensure_ascii=False, indent=2)

print(f"✅ {len(queries_data)} query uji disimpan ke: {queries_path}")
print()
for q in queries_data:
    ids = q["ground_truth_case_ids_same_pasal"]
    print(f"  {q['query_id']}")
    print(f"    pasal          : {q['ground_truth_pasal']}")
    print(f"    match di train : {ids if ids else '⚠️  tidak ada kasus dengan pasal sama di train set'}")


✅ 6 query uji disimpan ke: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\eval\queries.json
  query_case_028 -> ground truth pasal: Pasal 28
  query_case_016 -> ground truth pasal: Pasal 2 ayat (1)
  query_case_024 -> ground truth pasal: Pasal 27 ayat (1) Jo Pasal 45 ayat (1)
  query_case_018 -> ground truth pasal: Pasal 28 ayat (1) jo. Pasal 45
  query_case_009 -> ground truth pasal: Pasal 2
  query_case_010 -> ground truth pasal: Pasal 2 ayat (1)


In [ ]:
# ── Loop uji: jalankan retrieve() untuk semua query ──────────────
print("=" * 70)
print("UJI RETRIEVE UNTUK SEMUA QUERY (Jalur A: TF-IDF + Jalur B: Embedding)")
print("=" * 70)

for q in queries_data:
    qtext   = q["query_text"]
    pasal_gt = q["ground_truth_pasal"]
    ids_gt  = q["ground_truth_case_ids_same_pasal"]

    top_tfidf = retrieve_tfidf(qtext, k=5)
    top_bert  = retrieve_embedding(qtext, k=5)

    # Pasal dari hasil retrieval
    def get_pasals(case_ids):
        return train_df[train_df["case_id"].isin(case_ids)]["pasal_terbukti"].tolist()

    pasal_tfidf = get_pasals(top_tfidf)
    pasal_bert  = get_pasals(top_bert)

    # Apakah ada kasus dengan pasal cocok di top-5?
    hit_tfidf = any(p == pasal_gt for p in pasal_tfidf)
    hit_bert  = any(p == pasal_gt for p in pasal_bert)

    print(f"\n{q['query_id']}")
    print(f"  Ground truth pasal : {pasal_gt}")
    print(f"  Ground truth IDs   : {ids_gt}")
    print(f"  TF-IDF top-5 IDs   : {top_tfidf}  → pasal: {pasal_tfidf}  {'✅ HIT' if hit_tfidf else '❌ MISS'}")
    print(f"  BERT   top-5 IDs   : {top_bert}  → pasal: {pasal_bert}  {'✅ HIT' if hit_bert else '❌ MISS'}")


## 7. Simpan Model & Artefak ke Disk

Model-model yang sudah dilatih disimpan ke disk agar Tahap 4 (`04_predict.ipynb`) dan
Tahap 5 (`05_evaluation.ipynb`) bisa langsung memuatnya tanpa perlu re-run Tahap 3.

| Artefak | File | Kegunaan |
|---|---|---|
| `tfidf_vectorizer` | `data/models/tfidf_vectorizer.joblib` | Vektorisasi query baru di Tahap 4/5 |
| `svm_model` | `data/models/svm_model.joblib` | Klasifikasi pasal di Tahap 4/5 |
| `nb_model` | `data/models/nb_model.joblib` | Klasifikasi pasal alternatif |
| `train_embeddings` | `data/models/train_embeddings.npy` | Embedding BERT case base untuk cosine similarity |
| `train_df` | `data/models/train_df.csv` | Mapping index → case_id untuk retrieval |


In [ ]:
import joblib

MODELS_DIR = BASE_DIR / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Jalur A: TF-IDF + ML models
joblib.dump(tfidf_vectorizer, MODELS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(svm_model,        MODELS_DIR / "svm_model.joblib")
joblib.dump(nb_model,         MODELS_DIR / "nb_model.joblib")
print(f"✅ TF-IDF vectorizer + SVM + NB disimpan ke {MODELS_DIR}")

# Jalur B: BERT embeddings
np.save(MODELS_DIR / "train_embeddings.npy", train_embeddings)
print(f"✅ Train embeddings (shape: {train_embeddings.shape}) disimpan ke {MODELS_DIR}")

# Train set index (mapping index → case_id)
train_df.to_csv(MODELS_DIR / "train_df.csv", index=False)
print(f"✅ train_df.csv disimpan ke {MODELS_DIR}")

print()
print("Semua artefak Tahap 3 siap dimuat oleh Tahap 4 & 5.")


✅ TF-IDF vectorizer + SVM + NB disimpan ke C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\models
✅ Train embeddings (shape: (24, 768)) disimpan ke C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\models
✅ train_df.csv disimpan ke C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\models

Semua artefak Tahap 3 siap dimuat oleh Tahap 4 & 5.


## 8. Catatan & Langkah Selanjutnya

**Output Tahap 3 yang dihasilkan:**
- ✅ Fungsi `retrieve_tfidf()` dan `retrieve_embedding()`, keduanya teruji pada semua query
- ✅ File `data/eval/queries.json` — ground truth berbasis **pasal** (bukan case_id), karena  
  dokumen test set tidak ada di case base sehingga exact case_id match tidak relevan
- ✅ Model tersimpan di `data/models/`: `tfidf_vectorizer.joblib`, `svm_model.joblib`,  
  `nb_model.joblib`, `train_embeddings.npy`, `train_df.csv`
